# Week 7 Assignment - Delta Lake MERGE Implementation

**Objective:** Perform incremental data processing using Delta Lake.

**Steps:**
1. Load dataset into a Delta table
2. Perform basic cleaning (handle nulls, remove duplicates)
3. Create a second dataset simulating new/incremental data
4. Apply MERGE operation to update existing and insert new records
5. Validate results (row count, duplicates)
6. Display final dataset and summary


# Step 1 - Importing Libraries

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum,trim,count, lit



## Step 2 - Loading the Dataset into Delta table

In [0]:
master_path = "/Volumes/workspace/default/assignment_data/customer_master.csv"
incr_path = "/Volumes/workspace/default/assignment_data/customer_incremental.csv"

In [0]:
delta_table = "workspace.default.customer_master"

master_df = spark.read.csv(master_path, header=True, inferSchema=True)
display(master_df)

customer_id,name,city,email,signup_date
595,Rahul Ghosh,Ranchi,rahul.ghosh595@example.com,2023-07-10
548,Neha Kapoor,Guwahati,neha.kapoor548@example.com,2023-08-25
235,Damini Malhotra,Chennai,damini.malhotra235@example.com,2023-09-10
807,Farhan Das,Hyderabad,farhan.das807@example.com,2023-06-14
200,Arjun Rao,Coimbatore,arjun.rao200@example.com,2023-05-12
868,Payal Sinha,Jaipur,payal.sinha868@example.com,2023-05-11
80,Anjali Malhotra,Surat,anjali.malhotra80@example.com,2023-09-19
424,Rekha Ghosh,Visakhapatnam,rekha.ghosh424@example.com,2023-08-12
319,Ritu Kaur,Bengaluru,ritu.kaur319@example.com,2023-02-14
573,Alok Jain,Kochi,alok.jain573@example.com,2023-04-12


# Step 3 - Exploring Data

In [0]:
display(master_df.limit(10))

customer_id,name,city,email,signup_date
595,Rahul Ghosh,Ranchi,rahul.ghosh595@example.com,2023-07-10
548,Neha Kapoor,Guwahati,neha.kapoor548@example.com,2023-08-25
235,Damini Malhotra,Chennai,damini.malhotra235@example.com,2023-09-10
807,Farhan Das,Hyderabad,farhan.das807@example.com,2023-06-14
200,Arjun Rao,Coimbatore,arjun.rao200@example.com,2023-05-12
868,Payal Sinha,Jaipur,payal.sinha868@example.com,2023-05-11
80,Anjali Malhotra,Surat,anjali.malhotra80@example.com,2023-09-19
424,Rekha Ghosh,Visakhapatnam,rekha.ghosh424@example.com,2023-08-12
319,Ritu Kaur,Bengaluru,ritu.kaur319@example.com,2023-02-14
573,Alok Jain,Kochi,alok.jain573@example.com,2023-04-12


In [0]:
# Schema
master_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)
 |-- signup_date: date (nullable = true)



In [0]:
print("Row count:", master_df.count())
print("Column count:", len(master_df.columns))


Row count: 1050
Column count: 5


In [0]:
# Column Names
master_df.columns

['customer_id', 'name', 'city', 'email', 'signup_date']

## Step 4 - Cleaning Data(handling duplicates and null values)

#### Handeling Duplicate Values


In [0]:
print("Row count before cleaning:",master_df.count())

Row count before cleaning: 1050


In [0]:

duplicate_count = master_df.count() - master_df.distinct().count()
print("Duplicate Rows:",duplicate_count)

Duplicate Rows: 50


In [0]:
clean_df = master_df.dropDuplicates(["customer_id"])


In [0]:
# Verification
duplicate_count = clean_df.count() - clean_df.distinct().count()
print("Duplicate Rows:",duplicate_count)

Duplicate Rows: 0


###  Handeling Null Values


In [0]:

null_values_count = clean_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in clean_df.columns])
display(null_values_count)

customer_id,name,city,email,signup_date
0,25,0,40,0


In [0]:
clean_df = clean_df.dropna(subset="name")


In [0]:
clean_df = (clean_df
            .filter(trim(col("name")) != "")
            .fillna({"email": "not_provided"}))

In [0]:
print("Row count after cleaning:",clean_df.count())

Row count after cleaning: 975


In [0]:
clean_df.write.format("delta").mode("overwrite")\
    .option("overwriteSchema", "true").saveAsTable(delta_table)
print("Clean Delta table saved.")

Clean Delta table saved.


In [0]:
check_df = spark.table(delta_table)
print("Row count:", check_df.count())

Row count: 975


## Step 5 - Loading the incremental dataset 

In [0]:
incremental_df = spark.read.csv(incr_path, header=True, inferSchema=True)
display(incremental_df)

customer_id,name,city,email,signup_date
488,Indira Bhat,Bhopal,indira.bhat488_new@example.com,2023-10-10
212,Nikhil Nair,Vadodara,nikhil.nair212_new@example.com,2023-04-14
490,Sunil Mehta,Bengaluru,sunil.mehta490_new@example.com,2023-04-22
1049,Rahul Jain,Bengaluru,rahul.jain1049@example.com,2023-03-11
150,Priya Patel,Lucknow,priya.patel150_new@example.com,2023-10-28
423,Vivek Nair,Patna,vivek.nair423_new@example.com,2023-04-23
516,Ananya Joshi,Kanpur,ananya.joshi516_new@example.com,2023-06-15
496,Divya Joshi,Ahmedabad,divya.joshi496_new@example.com,2023-08-13
371,Sanjay Fernandes,Guwahati,sanjay.fernandes371_new@example.com,2023-08-24
588,Shreya Gupta,Lucknow,shreya.gupta588_new@example.com,2023-11-27


## Step 6 - MERGE operation (SCD Type 1: update the existing one and insert new)

In [0]:
incremental_df.createOrReplaceTempView("incremental_updates")
print("Temp view 'incremental_updates' ready.")

Temp view 'incremental_updates' ready.


In [0]:
spark.sql("""
MERGE INTO workspace.default.customer_master AS target
USING incremental_updates AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
print("MERGE completed.")

display(spark.table(delta_table).orderBy("customer_id"))

MERGE completed.


customer_id,name,city,email,signup_date
1,Rekha Nair,Jaipur,rekha.nair1@example.com,2023-12-18
2,Rajesh Sinha,Bengaluru,rajesh.sinha2@example.com,2023-12-13
3,Bhavna Desai,Kanpur,bhavna.desai3@example.com,2023-02-28
4,Nikhil Verma,Jaipur,nikhil.verma4@example.com,2023-02-16
5,Kavya Chopra,Visakhapatnam,not_provided,2023-01-27
6,Pooja Chauhan,Guwahati,pooja.chauhan6@example.com,2023-12-27
7,Isha Sinha,Kochi,isha.sinha7_new@example.com,2023-10-18
8,Amit Fernandes,Hyderabad,amit.fernandes8@example.com,2023-12-23
9,Deepak Das,Raipur,deepak.das9_new@example.com,2023-04-20
10,Divya Gupta,Surat,divya.gupta10_new@example.com,2023-02-21


## Step 7 - Validating Results(Checking row count and duplicates)

In [0]:
final_df = spark.table(delta_table)
print("Original Row count:",master_df.count())
print("Final Row count:", final_df.count())

dupes = (final_df.groupBy("customer_id")
    .agg(count(lit(1)).alias("cnt"))
    .filter(col("cnt") > 1)
)
print("Duplicate customer_ids:", dupes.count())
display(dupes)

Original Row count: 1050
Final Row count: 1078
Duplicate customer_ids: 0


customer_id,cnt


## Step 8 - Displaying final **Dataset**

In [0]:
print("Final Dataset:")
display(final_df.orderBy("customer_id"))

Final Dataset:


customer_id,name,city,email,signup_date
1,Rekha Nair,Jaipur,rekha.nair1@example.com,2023-12-18
2,Rajesh Sinha,Bengaluru,rajesh.sinha2@example.com,2023-12-13
3,Bhavna Desai,Kanpur,bhavna.desai3@example.com,2023-02-28
4,Nikhil Verma,Jaipur,nikhil.verma4@example.com,2023-02-16
5,Kavya Chopra,Visakhapatnam,not_provided,2023-01-27
6,Pooja Chauhan,Guwahati,pooja.chauhan6@example.com,2023-12-27
7,Isha Sinha,Kochi,isha.sinha7_new@example.com,2023-10-18
8,Amit Fernandes,Hyderabad,amit.fernandes8@example.com,2023-12-23
9,Deepak Das,Raipur,deepak.das9_new@example.com,2023-04-20
10,Divya Gupta,Surat,divya.gupta10_new@example.com,2023-02-21


In [0]:
print("Total customers:", final_df.count())


Total customers: 1078


In [0]:
print("Distinct cities:", final_df.select("city").distinct().count())


Distinct cities: 25


In [0]:
display(final_df.groupBy("city").count().orderBy("city"))

city,count
Ahmedabad,41
Bengaluru,46
Bhopal,53
Chandigarh,46
Chennai,60
Coimbatore,37
Delhi,40
Guwahati,52
Hyderabad,39
Indore,51


# Step 9 - Saving the final Dataset


In [0]:
output_path = "/Volumes/workspace/default/assignment_data/final_customer_output"

(final_df.orderBy("customer_id")
 .coalesce(1)                      
 .write.mode("overwrite")
 .option("header", True)
 .csv(output_path))

print("File saved to:", output_path)

File saved to: /Volumes/workspace/default/assignment_data/final_customer_output


## Delta Lake history - ACID & versioning evidence

In [0]:
display(spark.sql("DESCRIBE HISTORY workspace.default.customer_master"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
33,2026-07-06T00:52:39.000Z,72941889150098,kinshuworkspace@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1853100252392977),9bc54150-f698-437e-82a2-98891d55bd39,0706-003001-w7w2kcv5-v2n,32,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 24944, p25FileSize -> 19175, numDeletionVectorsRemoved -> 1, minFileSize -> 19175, numAddedFiles -> 1, maxFileSize -> 19175, p75FileSize -> 19175, p50FileSize -> 19175, numAddedBytes -> 19175)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
32,2026-07-06T00:52:37.000Z,72941889150098,kinshuworkspace@gmail.com,MERGE,"Map(predicate -> [""(customer_id#20358 = customer_id#20347)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1853100252392977),9bc54150-f698-437e-82a2-98891d55bd39,0706-003001-w7w2kcv5-v2n,31,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 7911, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 197, executionTimeMs -> 3260, materializeSourceTimeMs -> 196, numTargetRowsInserted -> 103, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1271, numTargetRowsUpdated -> 197, numOutputRows -> 300, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 300, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1763)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
31,2026-07-06T00:52:27.000Z,72941889150098,kinshuworkspace@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1853100252392977),690fe300-4d0a-423a-8c8d-0f9fe70007ef,0706-003001-w7w2kcv5-v2n,30,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 19175, numDeletionVectorsRemoved -> 0, numOutputRows -> 975, numOutputBytes -> 17033)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
30,2026-07-06T00:46:42.000Z,72941889150098,kinshuworkspace@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1853100252392977),3ad46655-44a1-4c18-a7ae-443ba7a7ac7d,0706-003001-w7w2kcv5-v2n,29,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 24944, p25FileSize -> 19175, numDeletionVectorsRemoved -> 1, minFileSize -> 19175, numAddedFiles -> 1, maxFileSize -> 19175, p75FileSize -> 19175, p50FileSize -> 19175, numAddedBytes -> 19175)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
29,2026-07-06T00:46:39.000Z,72941889150098,kinshuworkspace@gmail.com,MERGE,"Map(predicate -> [""(customer_id#18693 = customer_id#18682)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1853100252392977),3ad46655-44a1-4c18-a7ae-443ba7a7ac7d,0706-003001-w7w2kcv5-v2n,28,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 7911, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 197, executionTimeMs -> 3508, materializeSour

## Summary & Insights

- Loaded a **customer master** dataset (**1050 rows**) into a Delta table.
- **Cleaning** removed **50 duplicate rows** (deduped on customer_id) and null-name records, and filled null emails with `not_provided` - **975 clean rows**.
- Loaded an **incremental** dataset (**300 rows**) simulating new and updated customers.
- Applied **MERGE** (upsert): existing customers updated (SCD Type 1), new customers inserted - **1078 total rows**.
- **Validation** confirmed **zero duplicate customer IDs** in the final table.
- `DESCRIBE HISTORY` shows separate Delta versions for the write and the merge, demonstrating **ACID guarantees** and **time travel**.

